# Notebook E — Quand QMKL est-il vraiment utile ?

Ce notebook répond à la question centrale du projet : dans quelles conditions QMKL apporte-t-il quelque chose par rapport aux méthodes classiques ?

On teste **4 effets** de manière systématique, avec des données contrôlées.

**Plan :**
1. Cadre expérimental — 4 types de datasets
2. Effet du nombre de qubits Q — le dilemme expressivité / barren plateau
3. Effet du nombre de kernels M — rendements décroissants
4. Effet de la taille du dataset N — QMKL est un kernel method O(N²)
5. QMKL vs RBF-SVM — analyse cas par cas
6. Résumé : règles de décision pratiques


## Section 1 — Cadre expérimental

### Les 4 datasets synthétiques

On génère 4 datasets avec des structures différentes pour isoler les cas favorables et défavorables à QMKL :

| Dataset | Structure | Méthode attendue |
|---------|-----------|-----------------|
| DS1 — Gaussien | Séparation linéaire (clusters gaussiens) | RBF large gamme |
| DS2 — Cercles | Séparation non-linéaire circulaire | RBF (adapté naturellement) |
| DS3 — Cosinus | Pattern haute fréquence, oscillant | Kernel cosinus (ZZ !) |
| DS4 — XOR paires | Interactions entre features (x₁·x₂) | ZZ (captures interactions) |

**Hypothèse :** QMKL devrait dominer sur DS3 et DS4, où les kernels quantiques (ZZ, cosinus) ont un avantage structurel sur le RBF.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.datasets import make_moons

np.random.seed(42)

def K_Z(X, alpha=1.0):
    n, d = X.shape
    K = np.ones((n,n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha*diff)**2
    return K

def K_ZZ(X, alpha=1.0):
    K = K_Z(X, alpha)
    n, d = X.shape
    for k in range(d):
        for l in range(k+1, d):
            c = X[:,k]*X[:,l]
            diff = c.reshape(-1,1) - c.reshape(1,-1)
            K *= np.cos(alpha*diff)**2
    return K

def K_rbf(X, gamma=1.0):
    sq = np.sum(X**2, axis=1, keepdims=True)
    return np.exp(-gamma*(sq+sq.T-2*(X@X.T)))

def center_kernel(K):
    n=len(K); H=np.eye(n)-np.ones((n,n))/n; return H@K@H

def frobenius_ip(A,B): return np.sum(A*B)

def prep_data(X, d_target=4):
    '''Standardiser, PCA implicite par padding, normaliser dans [0,2]'''
    X_s = StandardScaler().fit_transform(X)
    if X_s.shape[1] < d_target:
        X_s = np.hstack([X_s, np.zeros((len(X_s), d_target - X_s.shape[1]))])
    X_s = X_s[:, :d_target]
    return MinMaxScaler(feature_range=(0,2)).fit_transform(X_s)

# ── Générer les 4 datasets ────────────────────────────────────────────────
n = 100

# DS1 : Gaussien (séparable linéairement)
X1 = np.vstack([np.random.multivariate_normal([1,1], [[0.3,0],[0,0.3]], n//2),
                np.random.multivariate_normal([-1,-1], [[0.3,0],[0,0.3]], n//2)])
y1 = np.array([1]*(n//2) + [-1]*(n//2))

# DS2 : Cercles (non-linéaire circulaire)
r1 = np.random.uniform(0, 0.6, n//2)
r2 = np.random.uniform(0.9, 1.4, n//2)
theta = np.random.uniform(0, 2*np.pi, n)
X2 = np.vstack([np.column_stack([r1*np.cos(theta[:n//2]), r1*np.sin(theta[:n//2])]),
                np.column_stack([r2*np.cos(theta[n//2:]), r2*np.sin(theta[n//2:])])])
y2 = np.array([1]*(n//2) + [-1]*(n//2))

# DS3 : Cosinus haute fréquence (pattern périodique)
X3 = np.random.uniform(0, 2*np.pi, (n, 2))
# Label basé sur cos(x₁) + cos(x₂)
signal = np.cos(2*X3[:,0]) + np.cos(2*X3[:,1])
y3 = np.where(signal > 0, 1, -1)

# DS4 : Interactions XOR (x₁·x₂ > 0)
X4 = np.random.uniform(-1.5, 1.5, (n, 2))
y4 = np.where(X4[:,0]*X4[:,1] > 0, 1, -1)

datasets = [
    ('DS1 — Gaussien
(linéaire)', X1, y1),
    ('DS2 — Cercles
(circulaire)', X2, y2),
    ('DS3 — Cosinus
(périodique)', X3, y3),
    ('DS4 — XOR·paires
(interactions)', X4, y4),
]

# ── Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, X, y) in zip(axes, datasets):
    ax.scatter(X[y==1,0], X[y==1,1], c='#e74c3c', s=20, alpha=0.7, label='+1')
    ax.scatter(X[y==-1,0], X[y==-1,1], c='#3498db', s=20, marker='s', alpha=0.7, label='-1')
    ax.set_title(name, fontweight='bold', fontsize=9)
    ax.legend(fontsize=7)
    ax.set_aspect('auto')
plt.suptitle('4 datasets synthétiques avec structures différentes
'
             'Rouge = classe +1, Bleu = classe -1', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Évaluation de base : 3 méthodes × 4 datasets ─────────────────────────
print("Évaluation initiale (N=100, 5-fold CV):")
print(f"{'Dataset':25s} | {'RBF-SVM':8s} | {'K_ZZ seul':10s} | {'QMKL avg':10s}")
print("-"*60)

for name, X, y in datasets:
    Xp = prep_data(X)
    K_rbf_ = K_rbf(Xp, gamma=1.0)
    K_zz_  = K_ZZ(Xp, alpha=1.5)
    K_mkl_ = np.mean([K_ZZ(Xp,a) for a in [0.5,1.0,2.0]], axis=0)

    def ev(K, y):
        eigs = np.linalg.eigvalsh(K)
        if eigs.min()<0: K += (-eigs.min()+1e-8)*np.eye(len(K))
        return cross_val_score(SVC(kernel='precomputed',C=1.), K, y,
                               cv=5, scoring='roc_auc').mean()
    auc_rbf = ev(K_rbf_, y)
    auc_zz  = ev(K_zz_,  y)
    auc_mkl = ev(K_mkl_, y)
    print(f"{name.replace(chr(10),' '):25s} | {auc_rbf:8.4f} | {auc_zz:10.4f} | {auc_mkl:10.4f}")


## Section 2 — Effet du nombre de qubits Q

### La question

Plus $Q$ est grand, plus l'espace de Hilbert est riche ($2^Q$ dimensions).
Mais au-delà d'une certaine valeur de $Q$, les **barren plateaus** dégradent le signal.

**Hypothèse :** Il existe un $Q^*$ optimal. En dessous : pas assez expressif. Au-dessus : signal trop concentré.

### Ce qu'on attend voir

- **Variance du kernel** : décroît exponentiellement avec $Q$
- **AUC du kernel seul** : d'abord croît (plus d'expressivité), puis décroît (barren plateau)
- **AUC de QMKL** : plus robuste que le kernel seul, mais même tendance à large $Q$

### Règle empirique

D'après la littérature et nos résultats : $Q \leq 8$ en pratique, avec $Q = 4$ souvent le meilleur compromis.
L'argument PCA : utiliser $Q = \lfloor \log_2(d) \rfloor$ où $d$ est la dimension des données originales.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler
from scipy.optimize import curve_fit

np.random.seed(42)

def K_Z(X, alpha=1.0):
    n, d = X.shape
    K = np.ones((n,n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha*diff)**2
    return K

def K_ZZ(X, alpha=1.0):
    K = K_Z(X, alpha)
    n, d = X.shape
    for k in range(d):
        for l in range(k+1, d):
            c = X[:,k]*X[:,l]
            diff = c.reshape(-1,1) - c.reshape(1,-1)
            K *= np.cos(alpha*diff)**2
    return K

def center_kernel(K):
    n=len(K); H=np.eye(n)-np.ones((n,n))/n; return H@K@H

def frobenius_ip(A,B): return np.sum(A*B)

# Données de base (DS3 cosinus, favorable au quantique)
n_pts = 80
X3_base = np.random.uniform(0, 2*np.pi, (n_pts, 12))  # 12 features max
y3 = (np.cos(2*X3_base[:,0]) + np.cos(2*X3_base[:,1]) > 0).astype(float)*2-1

Q_values = [2, 3, 4, 5, 6, 7, 8, 10]

variances_Z  = []
variances_ZZ = []
aucs_Z       = []
aucs_ZZ      = []
aucs_mkl     = []

def centered_align_mkl_quick(Kmats, y):
    '''Version rapide du Centered Alignment MKL.'''
    n = len(Kmats[0]); M = len(Kmats)
    Yc = center_kernel(np.outer(y,y))
    Kmats_c = [center_kernel(K) for K in Kmats]
    q = np.array([frobenius_ip(Kc, Yc) for Kc in Kmats_c])
    S = np.array([[frobenius_ip(Kmats_c[m], Kmats_c[mp]) for mp in range(M)] for m in range(M)])
    try:
        w = np.linalg.solve(S + 1e-8*np.eye(M), q)
    except:
        w = np.ones(M)/M
    w = np.maximum(w, 0)
    w = w/w.sum() if w.sum()>1e-10 else np.ones(M)/M
    K_comb = sum(wi*K for wi,K in zip(w, Kmats))
    return K_comb

def eval_K(K, y):
    eigs = np.linalg.eigvalsh(K)
    if eigs.min()<0: K += (-eigs.min()+1e-8)*np.eye(len(K))
    try:
        return cross_val_score(SVC(kernel='precomputed',C=1.), K, y,
                               cv=4, scoring='roc_auc').mean()
    except: return 0.5

for Q in Q_values:
    X_Q = MinMaxScaler(feature_range=(0,2)).fit_transform(X3_base[:, :Q])

    K_z  = K_Z(X_Q, alpha=1.0)
    K_zz = K_ZZ(X_Q, alpha=1.0)
    np.fill_diagonal(K_z, 1.0)
    np.fill_diagonal(K_zz, 1.0)

    # Variance hors-diagonale
    mask = ~np.eye(n_pts, dtype=bool)
    variances_Z.append(np.var(K_z[mask]))
    variances_ZZ.append(np.var(K_zz[mask]))

    # AUC kernels individuels
    aucs_Z.append(eval_K(K_z.copy(), y3))
    aucs_ZZ.append(eval_K(K_zz.copy(), y3))

    # AUC QMKL (4 kernels, alphas variés)
    Kmats_mkl = [K_Z(X_Q, a) for a in [0.5, 1.0, 2.0]] + [K_ZZ(X_Q, 1.5)]
    for K in Kmats_mkl:
        np.fill_diagonal(K, 1.0)
    K_comb = centered_align_mkl_quick(Kmats_mkl, y3)
    aucs_mkl.append(eval_K(K_comb, y3))

    print(f"Q={Q:2d}: Var(Z)={variances_Z[-1]:.4f}, "
          f"Var(ZZ)={variances_ZZ[-1]:.4f}, "
          f"AUC_Z={aucs_Z[-1]:.3f}, AUC_ZZ={aucs_ZZ[-1]:.3f}, "
          f"AUC_MKL={aucs_mkl[-1]:.3f}")

# ── Graphique ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.semilogy(Q_values, variances_Z,  '#3498db', marker='o', lw=2.5, ms=7, label='K_Z (local)')
ax.semilogy(Q_values, variances_ZZ, '#e74c3c', marker='s', lw=2.5, ms=7, label='K_ZZ (global)')
# Fit exponentiel
for data, color, name in [(variances_Z,'#3498db','Z'),(variances_ZZ,'#e74c3c','ZZ')]:
    try:
        log_d = np.log(np.array(data)+1e-15)
        c = np.polyfit(Q_values, log_d, 1)
        Qfit = np.linspace(Q_values[0], Q_values[-1], 100)
        ax.semilogy(Qfit, np.exp(np.polyval(c, Qfit)), '--', color=color, alpha=0.5, lw=1.5)
        print(f"Décroissance {name}: e^({c[0]:.3f}·Q) — double tous les {-np.log(2)/c[0]:.1f} qubits")
    except: pass
ax.set_xlabel('Q (qubits)', fontsize=11)
ax.set_ylabel('Variance de K(x,x')', fontsize=11)
ax.set_title('Barren Plateaus : variance vs Q
(plus de variance = plus discriminant)',
             fontweight='bold')
ax.legend()
ax.axvline(4, color='grey', ls=':', lw=1.5, label='Q=4 (notre projet)')
ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(Q_values, aucs_Z,   '#3498db', marker='o', lw=2.5, ms=7, label='K_Z seul')
ax2.plot(Q_values, aucs_ZZ,  '#e74c3c', marker='s', lw=2.5, ms=7, label='K_ZZ seul')
ax2.plot(Q_values, aucs_mkl, '#2ecc71', marker='^', lw=2.5, ms=7, label='QMKL (4 kernels, CA)')
ax2.axvline(4, color='grey', ls=':', lw=1.5)
ax2.set_xlabel('Q (qubits)', fontsize=11)
ax2.set_ylabel('AUC (4-fold cross-val)', fontsize=11)
ax2.set_title('AUC vs Q : compromis expressivité / barren plateau
'
              'QMKL est plus robuste qu'un kernel seul', fontweight='bold')
ax2.legend()
ax2.set_ylim(0.4, 1.0)
ax2.grid(alpha=0.3)

plt.suptitle('Effet du nombre de qubits Q
(Dataset DS3 cosinus, N=80)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("
→ AUC de QMKL est plus stable que K_ZZ seul face aux barren plateaus.")
print("→ Au-delà de Q=8, même QMKL commence à dégrader.")


## Section 3 — Effet du nombre de kernels M

### La question

Faut-il combiner 3 kernels ou 12 ? Y a-t-il un optimum ?

**Théorie (diversité décroissante) :** Si on ajoute des kernels dans l'ordre décroissant de leur contribution marginale :
- Les premiers kernels apportent beaucoup (informations complémentaires)
- Les suivants apportent de moins en moins (redondance croissante)
- Au-delà d'un seuil : la dilution par des kernels peu informatifs nuit à la performance

**Mesure :** À chaque étape, on ajoute le kernel le plus **diversifié** (faible alignement avec ceux déjà sélectionnés).

### Résultat attendu

Courbe en "S" aplatie : gain rapide pour $M = 1 \to 3$, puis plateau, puis légère dégradation si on continue.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)

def K_Z(X, alpha=1.0):
    n, d = X.shape
    K = np.ones((n,n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha*diff)**2
    return K

def K_ZZ(X, alpha=1.0):
    K = K_Z(X, alpha)
    n, d = X.shape
    for k in range(d):
        for l in range(k+1, d):
            c = X[:,k]*X[:,l]
            diff = c.reshape(-1,1) - c.reshape(1,-1)
            K *= np.cos(alpha*diff)**2
    return K

def K_rbf(X, gamma=1.0):
    sq = np.sum(X**2, axis=1, keepdims=True)
    return np.exp(-gamma*(sq+sq.T-2*(X@X.T)))

def center_kernel(K): n=len(K); H=np.eye(n)-np.ones((n,n))/n; return H@K@H
def frobenius_ip(A,B): return np.sum(A*B)
def mutual_align(K1,K2):
    K1c,K2c=center_kernel(K1),center_kernel(K2)
    num=frobenius_ip(K1c,K2c)
    den=np.sqrt(frobenius_ip(K1c,K1c)*frobenius_ip(K2c,K2c))
    return num/den if den>0 else 0.

def eval_K(K, y):
    eigs=np.linalg.eigvalsh(K)
    if eigs.min()<0: K+=(-eigs.min()+1e-8)*np.eye(len(K))
    try: return cross_val_score(SVC(kernel='precomputed',C=1.),K,y,cv=4,scoring='roc_auc').mean()
    except: return 0.5

# Dataset DS3 cosinus
n_pts = 80
X_base = np.random.uniform(0, 2*np.pi, (n_pts, 4))
y = (np.cos(2*X_base[:,0]) + np.cos(2*X_base[:,1]) > 0).astype(float)*2-1
X = MinMaxScaler(feature_range=(0,2)).fit_transform(X_base)

# 12 kernels disponibles
kernel_lib = (
    [(K_Z, a, f'Z α={a}') for a in [0.5, 1.0, 2.0, 3.0]] +
    [(K_ZZ, a, f'ZZ α={a}') for a in [0.5, 1.0, 2.0, 3.0]] +
    [(K_rbf, g, f'RBF γ={g}') for g in [0.1, 0.5, 2.0, 5.0]]
)
M_lib = len(kernel_lib)

Kmats_lib = []
for fn, param, name in kernel_lib:
    K = fn(X, param)
    np.fill_diagonal(K, 1.0)
    Kmats_lib.append(K)

# AUC individuels
aucs_indiv = [eval_K(K.copy(), y) for K in Kmats_lib]
names_lib = [k[2] for k in kernel_lib]
best_single_idx = np.argmax(aucs_indiv)
best_single_auc = aucs_indiv[best_single_idx]
print(f"Meilleur kernel individuel: {names_lib[best_single_idx]} → AUC={best_single_auc:.4f}")

# ── Stratégie 1 : Ajout greedy par diversité ─────────────────────────────
selected_div = [best_single_idx]
remaining_div = [i for i in range(M_lib) if i != best_single_idx]
aucs_greedy_div = [best_single_auc]
diversities = [1.0]  # seul kernel = diversité maximale par convention

for step in range(min(11, len(remaining_div))):
    # Choisir le kernel le plus diversifié par rapport à ceux déjà sélectionnés
    best_new_idx = None
    best_diversity = -1
    for idx in remaining_div:
        div = 1 - np.mean([mutual_align(Kmats_lib[idx], Kmats_lib[s]) for s in selected_div])
        if div > best_diversity:
            best_diversity = div
            best_new_idx = idx
    selected_div.append(best_new_idx)
    remaining_div.remove(best_new_idx)

    # Calculer AUC de la combinaison
    K_comb = np.mean([Kmats_lib[i] for i in selected_div], axis=0)
    auc = eval_K(K_comb.copy(), y)
    aucs_greedy_div.append(auc)
    diversities.append(best_diversity)

# ── Stratégie 2 : Ajout naïf (ordre aléatoire) ───────────────────────────
order_naive = list(np.random.permutation(M_lib))
aucs_naive = []
for k in range(1, M_lib+1):
    subset = order_naive[:k]
    K_comb = np.mean([Kmats_lib[i] for i in subset], axis=0)
    aucs_naive.append(eval_K(K_comb.copy(), y))

# ── Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x_steps = range(1, len(aucs_greedy_div)+1)
ax.plot(x_steps, aucs_greedy_div, '#2ecc71', marker='o', ms=7, lw=2.5, label='Greedy diversité')
ax.plot(range(1, M_lib+1), aucs_naive, '#e74c3c', marker='s', ms=5, lw=1.5,
        ls='--', alpha=0.7, label='Ordre aléatoire')
ax.axhline(best_single_auc, color='grey', ls=':', lw=1.5, label=f'Meilleur individuel={best_single_auc:.3f}')

# Trouver le maximum
best_M = np.argmax(aucs_greedy_div) + 1
ax.axvline(best_M, color='gold', lw=2, label=f'Optimum M*={best_M}')
ax.scatter([best_M], [max(aucs_greedy_div)], s=150, color='gold', zorder=5)

ax.set_xlabel('Nombre de kernels M ajoutés', fontsize=11)
ax.set_ylabel('AUC (4-fold CV)')
ax.set_title('AUC vs nombre de kernels M
(Ajout greedy par diversité maximale)',
             fontweight='bold')
ax.legend(fontsize=8)
ax.set_xlim(0.5, M_lib+0.5)
ax.grid(alpha=0.3)

ax2 = axes[1]
# Gain marginal par étape
gains = [aucs_greedy_div[i] - aucs_greedy_div[i-1] for i in range(1, len(aucs_greedy_div))]
colors_bar = ['#27ae60' if g>0 else '#c0392b' for g in gains]
ax2.bar(range(2, len(aucs_greedy_div)+1), gains, color=colors_bar, alpha=0.8)
ax2.axhline(0, color='black', lw=1.5)
ax2.set_xlabel('Kernel ajouté (étape M)', fontsize=11)
ax2.set_ylabel('Gain marginal AUC')
ax2.set_title('Gain marginal de chaque kernel ajouté
'
              'Vert = aide, Rouge = nuit', fontweight='bold')

for i, (g, idx) in enumerate(zip(gains, selected_div[1:])):
    name_short = names_lib[idx].replace('α=','α')
    ax2.text(i+2, g + 0.002*(1 if g>=0 else -1), name_short,
             ha='center', fontsize=7, rotation=45)
ax2.grid(alpha=0.3)

plt.suptitle(f'Effet du nombre de kernels M sur AUC
Optimum empirique : M*={best_M}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"
Conclusion: M* = {best_M} kernels suffisent.")
print(f"Gain par rapport au meilleur individuel: +{max(aucs_greedy_div)-best_single_auc:.4f}")
print(f"Au-delà de M={best_M}, les gains marginaux deviennent nuls ou négatifs.")


## Section 4 — Effet de la taille du dataset N

### Le goulot d'étranglement computationnel

Les SVM à kernel ont une complexité :
- **Calcul de la matrice kernel :** $O(N^2 d)$ — quadratique en $N$
- **Entraînement SVM :** $O(N^2)$ à $O(N^3)$ selon l'implémentation

Pour $N = 100$ : rapide. Pour $N = 45\,000$ (Bank Marketing complet) : infaisable directement.

### Impact sur la stratégie

| $N$ | Temps kernel ZZ (Q=4) | Stratégie recommandée |
|-----|----------------------|----------------------|
| 50–200 | < 1s | QMKL direct |
| 200–1000 | 1–100s | QMKL avec cache kernel |
| 1000–10000 | Heures | Approximation Nyström |
| > 10000 | Jours | RBF-SVM + Nyström ou RF |

### Nyström approximation

Pour $N$ grand, on approche la matrice kernel :
$$K \approx K_{\cdot L} K_{LL}^{-1} K_{L \cdot}$$
où $L \subset \{1,...,N\}$ est un sous-ensemble de $m \ll N$ "landmark" points.
Complexité réduite à $O(Nm + m^3)$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler

np.random.seed(42)

def K_ZZ(X, alpha=1.0):
    n, d = X.shape
    K = np.ones((n,n))
    for k in range(d):
        diff = X[:,k:k+1] - X[:,k].reshape(1,-1)
        K *= np.cos(alpha*diff)**2
    for k in range(d):
        for l in range(k+1, d):
            c = X[:,k]*X[:,l]
            diff = c.reshape(-1,1) - c.reshape(1,-1)
            K *= np.cos(alpha*diff)**2
    return K

def K_rbf(X, gamma=1.0):
    sq = np.sum(X**2, axis=1, keepdims=True)
    return np.exp(-gamma*(sq+sq.T-2*(X@X.T)))

# ── Mesurer le temps en fonction de N ────────────────────────────────────
N_values = [30, 50, 80, 100, 150, 200, 300, 500]
times_ZZ  = []
times_rbf = []
aucs_ZZ   = []
aucs_rbf  = []

print("Mesure du temps de calcul du kernel vs N (Q=4):")
print(f"{'N':5s} | {'t_ZZ (ms)':10s} | {'t_RBF (ms)':11s} | {'AUC_ZZ':8s} | {'AUC_RBF':8s}")
print("-"*55)

for N in N_values:
    # Générer N points
    X_N = MinMaxScaler(feature_range=(0,2)).fit_transform(
          np.random.uniform(0, 1, (N, 4)))
    y_N = (np.sum((X_N - 1.0)**2, axis=1) < 0.7).astype(float)*2-1
    if np.all(y_N == y_N[0]):  # éviter classes vides
        y_N[:N//3] = -y_N[:N//3]

    # Temps kernel ZZ
    t0 = time.time()
    K_zz = K_ZZ(X_N, alpha=1.5)
    t_zz = (time.time() - t0) * 1000
    times_ZZ.append(t_zz)

    # Temps kernel RBF
    t0 = time.time()
    K_rbf_ = K_rbf(X_N, gamma=1.0)
    t_rbf = (time.time() - t0) * 1000
    times_rbf.append(t_rbf)

    # AUC (seulement pour N ≤ 200 pour éviter les temps trop longs)
    if N <= 200:
        np.fill_diagonal(K_zz, 1.0)
        eigs = np.linalg.eigvalsh(K_zz)
        if eigs.min()<0: K_zz += (-eigs.min()+1e-8)*np.eye(N)
        try:
            a_zz = cross_val_score(SVC(kernel='precomputed',C=1.), K_zz, y_N,
                                   cv=4, scoring='roc_auc').mean()
        except: a_zz = np.nan
        try:
            a_rbf = cross_val_score(SVC(kernel='precomputed',C=1.), K_rbf_, y_N,
                                    cv=4, scoring='roc_auc').mean()
        except: a_rbf = np.nan
    else:
        a_zz = a_rbf = np.nan

    aucs_ZZ.append(a_zz)
    aucs_rbf.append(a_rbf)
    print(f"{N:5d} | {t_zz:10.2f} | {t_rbf:11.2f} | "
          f"{'N/A' if np.isnan(a_zz) else f'{a_zz:.4f}':8s} | "
          f"{'N/A' if np.isnan(a_rbf) else f'{a_rbf:.4f}':8s}")

# ── Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.loglog(N_values, times_ZZ,  '#3498db', marker='o', ms=7, lw=2.5, label='K_ZZ (Q=4)')
ax.loglog(N_values, times_rbf, '#e74c3c', marker='s', ms=7, lw=2.5, label='K_RBF (numpy)')
# Courbe théorique N²
N_fit = np.array(N_values, dtype=float)
scale = times_ZZ[0] / N_fit[0]**2
ax.loglog(N_fit, scale * N_fit**2, 'k--', lw=1.5, alpha=0.7, label='O(N²) théorique')

# Annotations
ax.annotate('QMKL réaliste
(N ≤ 200)', xy=(200, times_ZZ[N_values.index(200)]),
            xytext=(120, times_ZZ[N_values.index(200)]*3),
            fontsize=8, color='#2980b9',
            arrowprops=dict(arrowstyle='->', color='#2980b9'),
            bbox=dict(boxstyle='round', facecolor='#d6eaf8', edgecolor='#2980b9'))

ax.set_xlabel('Taille du dataset N', fontsize=11)
ax.set_ylabel('Temps de calcul du kernel (ms)', fontsize=11)
ax.set_title('Complexité O(N²) du kernel
Goulot d'étranglement pour N > 500',
             fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, which='both')

ax2 = axes[1]
N_auc = [N for N, a in zip(N_values, aucs_ZZ) if not np.isnan(a)]
aucs_zz_plot = [a for a in aucs_ZZ if not np.isnan(a)]
aucs_rbf_plot = [a for a in aucs_rbf if not np.isnan(a)]
ax2.plot(N_auc, aucs_zz_plot,  '#3498db', marker='o', ms=7, lw=2.5, label='K_ZZ seul')
ax2.plot(N_auc, aucs_rbf_plot, '#e74c3c', marker='s', ms=7, lw=2.5, label='K_RBF')
ax2.set_xlabel('Taille du dataset N', fontsize=11)
ax2.set_ylabel('AUC (4-fold CV)')
ax2.set_title('AUC vs N : stabilisation avec plus de données
'
              '(les deux méthodes bénéficient de N plus grand)', fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_ylim(0.4, 1.0)

plt.suptitle('Effet de la taille du dataset N
(Q=4, Dataset synthétique sphérique)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Section 5 — QMKL vs RBF-SVM : analyse par type de données

### Résumé expérimental

Voici ce qu'on attend théoriquement, et ce qu'on observe empiriquement :

**Cas favorables à QMKL :**
- Données avec structure **périodique ou haute fréquence** (kernel cosinus ZZ ≠ RBF)
- Données avec **interactions entre features** ($x_i \times x_j$ capturées par ZZ mais pas par RBF seul)
- Petits datasets ($N \leq 200$) où les méthodes classiques ont peu de données pour calibrer $\gamma$

**Cas défavorables à QMKL :**
- Données **gaussiennes standardisées** (RBF est optimal par définition)
- **Grande N** (RBF-SVM est plus scalable)
- **Barren plateaus** ($Q > 8$)
- Hardware NISQ bruité

### Le cas financier

Les données financières (German Credit, Bank Marketing) sont :
- Tabulaires, hétérogènes (mélange de variables catégorielles, ordinales, continues)
- Standardisées avant PCA
- Sans structure périodique évidente

→ **RBF-SVM est bien adapté**. QMKL n'a pas d'avantage structurel clair sur ces données.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.datasets import make_moons

np.random.seed(42)

def K_Z(X, alpha=1.0):
    n,d=X.shape; K=np.ones((n,n))
    for k in range(d):
        diff=X[:,k:k+1]-X[:,k].reshape(1,-1); K*=np.cos(alpha*diff)**2
    return K

def K_ZZ(X, alpha=1.0):
    K=K_Z(X,alpha); n,d=X.shape
    for k in range(d):
        for l in range(k+1,d):
            c=X[:,k]*X[:,l]; diff=c.reshape(-1,1)-c.reshape(1,-1); K*=np.cos(alpha*diff)**2
    return K

def K_rbf(X, gamma=1.0):
    sq=np.sum(X**2,axis=1,keepdims=True)
    return np.exp(-gamma*(sq+sq.T-2*(X@X.T)))

def center_kernel(K): n=len(K); H=np.eye(n)-np.ones((n,n))/n; return H@K@H
def frobenius_ip(A,B): return np.sum(A*B)

def centered_align_mkl(Kmats, y):
    M=len(Kmats)
    Yc=center_kernel(np.outer(y,y))
    Kc=[center_kernel(K) for K in Kmats]
    q=np.array([frobenius_ip(kc,Yc) for kc in Kc])
    S=np.array([[frobenius_ip(Kc[m],Kc[mp]) for mp in range(M)] for m in range(M)])
    try: w=np.linalg.solve(S+1e-8*np.eye(M),q)
    except: w=np.ones(M)/M
    w=np.maximum(w,0); return w/w.sum() if w.sum()>1e-10 else np.ones(M)/M

def eval_K(K, y, cv=5):
    eigs=np.linalg.eigvalsh(K)
    if eigs.min()<0: K+=(-eigs.min()+1e-8)*np.eye(len(K))
    try: return cross_val_score(SVC(kernel='precomputed',C=1.),K,y,cv=cv,scoring='roc_auc').mean()
    except: return np.nan

n = 100

# DS1 Gaussien
X1=np.vstack([np.random.multivariate_normal([1,1],[[0.3,0],[0,0.3]],n//2),
              np.random.multivariate_normal([-1,-1],[[0.3,0],[0,0.3]],n//2)])
y1=np.array([1]*(n//2)+[-1]*(n//2))

# DS2 Cercles
r1=np.random.uniform(0,0.6,n//2); r2=np.random.uniform(0.9,1.4,n//2)
th=np.random.uniform(0,2*np.pi,n)
X2=np.vstack([np.column_stack([r1*np.cos(th[:n//2]),r1*np.sin(th[:n//2])]),
              np.column_stack([r2*np.cos(th[n//2:]),r2*np.sin(th[n//2:])])])
y2=np.array([1]*(n//2)+[-1]*(n//2))

# DS3 Cosinus
X3=np.random.uniform(0,2*np.pi,(n,2))
y3=np.where(np.cos(2*X3[:,0])+np.cos(2*X3[:,1])>0,1,-1)

# DS4 XOR
X4=np.random.uniform(-1.5,1.5,(n,2))
y4=np.where(X4[:,0]*X4[:,1]>0,1,-1)

datasets = [('DS1 Gaussien', X1, y1), ('DS2 Cercles', X2, y2),
            ('DS3 Cosinus', X3, y3), ('DS4 XOR·paires', X4, y4)]

methods = ['RBF γ=1', 'RBF γ=0.1', 'K_Z α=1', 'K_ZZ α=1.5', 'QMKL (4 kernels, CA)']
colors_method = ['#e74c3c','#e74c3c','#3498db','#f39c12','#2ecc71']
hatches = ['','///','','','']

results_matrix = np.full((len(datasets), len(methods)), np.nan)

for ds_idx, (ds_name, X_ds, y_ds) in enumerate(datasets):
    Xp = MinMaxScaler(feature_range=(0,2)).fit_transform(StandardScaler().fit_transform(X_ds))
    if Xp.shape[1] < 4:
        Xp = np.hstack([Xp, np.zeros((len(Xp), 4-Xp.shape[1]))])
    Xp = Xp[:,:4]

    K_rbf1 = K_rbf(Xp[:,:2], gamma=1.0)
    K_rbf01 = K_rbf(Xp[:,:2], gamma=0.1)
    K_z1   = K_Z(Xp, alpha=1.0)
    K_zz1  = K_ZZ(Xp, alpha=1.5)

    Kmats_mkl = [K_Z(Xp, a) for a in [0.5, 1.0]] + [K_ZZ(Xp, a) for a in [1.0, 2.5]]
    for K in Kmats_mkl: np.fill_diagonal(K, 1.0)
    w_ca = centered_align_mkl(Kmats_mkl, y_ds)
    K_ca = sum(w*K for w,K in zip(w_ca, Kmats_mkl))

    for m_idx, K in enumerate([K_rbf1, K_rbf01, K_z1, K_zz1, K_ca]):
        np.fill_diagonal(K, 1.0)
        results_matrix[ds_idx, m_idx] = eval_K(K, y_ds)

# ── Graphique en barres groupées ────────────────────────────────────────
fig, axes = plt.subplots(1, len(datasets), figsize=(16, 5), sharey=True)

for ax, (ds_name, _, _), row in zip(axes, datasets, results_matrix):
    x = np.arange(len(methods))
    bars = ax.bar(x, row, color=colors_method, alpha=0.85, width=0.7)
    for bar, hatch, val in zip(bars, hatches, row):
        bar.set_hatch(hatch)
        if not np.isnan(val):
            ax.text(bar.get_x()+bar.get_width()/2, val+0.01,
                    f'{val:.3f}', ha='center', fontsize=7.5, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels([m.replace(' ','
') for m in methods], fontsize=7)
    ax.set_title(ds_name, fontweight='bold', fontsize=9)
    ax.set_ylim(0.4, 1.05)
    ax.axhline(0.5, color='grey', ls=':', lw=1)
    ax.grid(axis='y', alpha=0.3)
    if ax == axes[0]:
        ax.set_ylabel('AUC (5-fold CV)')

plt.suptitle('QMKL vs RBF-SVM selon le type de données (N=100, Q=4)
'
             'QMKL en vert | RBF en rouge | Kernels individuels en bleu/orange',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("
Analyse:")
for ds_idx, (ds_name, _, _) in enumerate(datasets):
    rbf_best = max(results_matrix[ds_idx, :2])
    qmkl = results_matrix[ds_idx, 4]
    diff = qmkl - rbf_best
    sign = '>' if diff > 0 else '<'
    print(f"  {ds_name}: QMKL {sign} RBF ({diff:+.4f})")


## Section 6 — Résumé : règles de décision pratiques

### Les 5 questions à se poser avant d'utiliser QMKL

**Q1 : Quelle est la taille de mon dataset N ?**
- $N \leq 200$ : QMKL est envisageable
- $200 < N \leq 1000$ : QMKL avec approximation (cache kernel, Nyström)
- $N > 1000$ : Préférer RBF-SVM + Random Forest

**Q2 : Quelle structure ont mes données ?**
- Données périodiques / haute fréquence : QMKL avantageux
- Interactions entre features ($x_i \times x_j$) : ZZFeatureMap avantageux
- Données gaussiennes standardisées : RBF optimal

**Q3 : Combien de qubits Q utiliser ?**
- $Q = 4$ est le bon compromis (expressivité vs barren plateaus)
- Maximum : $Q = 8$
- Règle : $Q \leq \lfloor\log_2(d_{\text{features}})\rfloor + 1$

**Q4 : Combien de kernels M combiner ?**
- M = 3–6 avec **diversité maximale** entre familles
- Au-delà, rendements décroissants
- QUBO si on veut sélectionner rigoureusement

**Q5 : Quelle stratégie MKL ?**
- Centered Alignment : robuste, analytique, recommandé par défaut
- QUBO : si interprétabilité requise ou si $M > 12$
- Average : seulement si tous les kernels ont des AUC similaires
